# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = "openai/gpt-4o"

BASE_URL = "https://models.github.ai/inference"

openai = OpenAI(base_url=BASE_URL, api_key=api_key)

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-4o
Found 2 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-4o
Found 3 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-4o
Found 3 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-OCR
Updated
5 days ago
•
204k
•
767
Qwen/Qwen3-Coder-Next
Updated
4 days ago
•
53.5k
•
572
moonshotai/Kimi-K2.5
Updated
3 days ago
•
335k
•
1.81k
stepfun-ai/Step-3.5-Flash
Updated
about 13 hours ago
•
12k
•
508
circlestone-labs/Anima
Updated
7 days ago
•
60.6k
•
497
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.29k
Qwen3-TTS Demo
🎙
1.29k
Transform text into natural-sounding speech with custom voices
Running
on
A100
175
ACE-Step v1.5
🎵
175
Music Generation Foundation Model v1.5
Running
467
Demo Playground
⚡
467
Fre

In [26]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-4o
Found 6 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-OCR\nUpdated\n5 days ago\n•\n204k\n•\n767\nQwen/Qwen3-Coder-Next\nUpdated\n4 days ago\n•\n53.5k\n•\n572\nmoonshotai/Kimi-K2.5\nUpdated\n3 days ago\n•\n335k\n•\n1.81k\nstepfun-ai/Step-3.5-Flash\nUpdated\nabout 13 hours ago\n•\n12k\n•\n508\ncirclestone-labs/Anima\nUpdated\n7 days ago\n•\n60.6k\n•\n497\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.29k\nQwen3-TTS Demo\n

In [27]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="openai/gpt-4o",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-4o
Found 6 relevant links


# Welcome to Hugging Face  

## Who We Are  
Hugging Face is a leading AI and machine learning (ML) platform empowering the global community to shape the future of AI. We serve as **The AI Community Building the Future**, enabling collaboration on models, datasets, and applications through our innovative platform. With over **2 million models, 1 million applications, and 500,000 datasets**, Hugging Face is the go-to hub for ML enthusiasts, practitioners, and enterprises working across all AI modalities – text, image, video, audio, 3D, and beyond.

Our mission extends beyond providing tools; we are a space where AI is democratized, making it more inclusive, accessible, and creative for everyone.

---

## For Our Customers  
Hugging Face offers a comprehensive suite of services to ensure customers of all scales can thrive in the world of AI. Key features include:  

- **Collaboration Platform:** Host and collaborate on public or private models, datasets, and applications.
- **Enterprise Solutions:** Unlock advanced AI functionalities with secure, scalable, and flexible enterprise-grade capabilities.
- **Open Source Stack:** Accelerate workflows with powerful and adaptable tools.
- **Seamless Integration:** Choose from tailored integrations, regions, and analytical dashboards that align with your business needs.
- **Compute Optimization:** Access advanced compute options like ZeroGPU to boost scalability and performance.  
- **Enterprise Customizations:** Benefit from features like single sign-on (SSO), priority support, granular access controls, and private storage.  

Whether you're an individual innovator or an enterprise team, Hugging Face equips you with the most advanced platform to build AI efficiently and responsibly.

---

## Company Culture & Community  
At Hugging Face, we embrace open collaboration, innovation, and inclusivity. Our platform is built to connect a vibrant community of developers, researchers, and enterprises who share ideas, push boundaries, and co-create cutting-edge tools. The values we promote are:  

- **Accessibility:** Bridging the gap between AI research and real-world applications.
- **Innovation:** Fostering a creative and forward-thinking environment to solve complex problems.
- **Community-Driven:** Supporting a collaborative culture where everyone's contributions matter.  

Our open-source approach and active engagement with the AI ecosystem underscore our belief in making AI technologies widely accessible for public benefit.

---

## Careers at Hugging Face  
Join our mission to shape the future of AI! At Hugging Face, we offer an energized and inclusive workplace where your ideas and efforts directly impact the world of machine learning. We're always looking for passionate individuals who share our vision to democratize AI.

Learn more about our current openings on our [Careers Page](#).

---

## Why Choose Hugging Face?  
- **Proven Expertise:** The trusted platform for millions of users across research, academia, and enterprises.
- **End-to-End Ecosystem:** Contribute to and benefit from a unified hub for models, datasets, and applications.
- **Scalable Solutions:** Manage growth with enterprise-grade tools and advanced compute options tailored to your needs.
- **Global Community Impact:** Be part of a thriving ML community collaborating to advance AI responsibly.

Hugging Face is more than a company. It's a movement bringing together the best minds in AI to ensure its future is inclusive, innovative, and impactful. Explore limitless AI possibilities with us!

For more information, visit: [Hugging Face Website](https://huggingface.co).

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="openai/gpt-4o",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        # Skip chunks with no choices or no delta/content
        if not chunk.choices or not hasattr(chunk.choices[0], "delta") or not hasattr(chunk.choices[0].delta, "content"):
            continue
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-4o
Found 7 relevant links


# Hugging Face  

## About Us  
Hugging Face is the leading AI community and platform that empowers the future of machine learning (ML). We are dedicated to creating a global hub where developers, researchers, and organizations can collaborate on ML models, datasets, and applications. With over **2 million models**, **1 million applications**, and **500,000 datasets**, Hugging Face is the ultimate destination for building, discovering, and sharing cutting-edge AI technologies.

Our platform emphasizes open collaboration, innovation, and the democratization of machine learning for individuals and enterprises alike. With Hugging Face, create transformative projects, explore diverse AI modalities, and build an impactful portfolio to showcase your expertise.

---

## Our Offerings  

### Models  
Access a vast library of over **2 million machine learning models**, including trending, state-of-the-art tools for various tasks like text generation, image creation, speech processing, and more.

### Datasets  
Explore over **500,000 datasets** to enhance your machine learning projects. Whether you’re a researcher or developer, these datasets fuel new innovations in AI.

### Applications & Spaces  
Discover more than **1 million AI applications** and AI Spaces that transform concepts into interactive tools. From text-to-speech demos to image generation, Hugging Face Spaces offer powerful solutions for everyone.

### Enterprise Solutions  
Our **Team & Enterprise Hub** provides organizations with the tools to scale their AI efforts effectively, with robust features like:  
- **Enterprise-Grade Security**: Advanced access controls and protection for your data.  
- **Single Sign-On Integration (SSO)**: Enhanced connectivity with your identity provider.  
- **Regional Data Management**: Configure and manage repository data based on your organizational needs.  
- **Comprehensive Audit Logs**: Track and maintain control over all actions taken.  
Subscribed plans start at **$20/user/month**, or tailored enterprise solutions can be created to meet your business needs.  

---

## Why Choose Hugging Face?  

### Collaboration at Its Best  
Our community-driven platform connects a global ecosystem of AI enthusiasts, researchers, and developers. Share your models and datasets publicly, gain feedback, and create innovative solutions together.  

### Open-Source Excellence  
Leverage the **Hugging Face Open Source Stack** to boost your development speed while enjoying access to pre-built tools and resources in text, image, video, audio, and even 3D.

### Innovation at Scale  
With Hugging Face, you can transform your ideas into practical AI applications using our cutting-edge technology, expertise, and an ever-growing library of open and shared resources.  

---

## Join the Hugging Face Community  

Joining Hugging Face means joining a global movement to democratize artificial intelligence. Whether you’re a developer passionate about machine learning, an enterprise innovating with AI, or an investor looking to empower the future of technology, Hugging Face welcomes you.  

### Career Opportunities  
Be part of a community that thrives on curiosity, innovation, and collaboration. As a fast-growing company in the AI space, we seek talented individuals eager to work on groundbreaking projects and shape the next era of artificial intelligence.

Explore exciting career opportunities at Hugging Face and join our mission to create a more collaborative AI-powered future.  

---

## Contact Us  
Ready to accelerate your machine learning journey with Hugging Face?  

- **Visit Us**: [www.huggingface.co](https://www.huggingface.co)  
- **Enterprise Sales**: Contact our sales team for tailored solutions.  
- **Join the Community**: Sign up to host your models, access datasets, and collaborate with other ML pioneers.  

Hugging Face – **The AI community building the future.**

In [29]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-4o
Found 3 relevant links


**Welcome to Hugging Face: Where Machine Learning Gets a Personality**

**The AI Community Building the Future—One Hug at a Time**  
At Hugging Face, we're not just talking about machine learning—we're hugging it, nurturing it, and letting it run free! Think of us as the most vibrant playroom for the world's AI thinkers, builders, and dreamers. From models that seem straight out of sci-fi to datasets that put the "intelligence" in AI, we’re designing the future... with a big dose of quirk and collaboration.  

Whether you're a seasoned code wizard, a curious newbie, or just someone fascinated by algorithms with cute names, dive in! We’ve got over **2 million (yes, million!) models** and **500,000+ datasets** just waiting for you to tinker with. Explore, collaborate, OR impress your boss (we won’t tell!).

---

**Why We’re Everyone’s New BFF in AI**  
1. **Collaboration Central**: Create, share, discover, and experiment on the **Hugging Face Hub**. Push the boundaries of ML together with the best minds in the world. Yes, Carl, this does mean you can casually say you're changing the world (finally).  
2. **Open Arms, Open Source**: Our community thrives on transparency and sharing. Because building THE future can only happen when we do it together.  
3. **All-in-One Playground**: Text, images, videos, audio—even 3D—our platform knows no bounds. (What’s next—AI generating gourmet pizzas? Give it time.)  
4. **Your AI Portfolio Awaits**: Show off your work and your skills. (Yes, we’re subtly saying you might even get famous. AI-famous, but still.)  

---

**Why Join Hugging Face?**  
Think of us as a tech startup with Silicon Valley dreams but European charm.  
Here’s what you can expect when you work with us:  
- A focus on building **ethical, open-source AI** with lasting value. No dystopian robot overlords allowed.  
- An **inclusive, collaborative vibe**. (We like to say we work hard, but we hug harder.)  
- Cutting-edge projects that will make you say, "Wait, I get paid to do this?"  
- Oh, and access to every quirky AI model under the sun. From text-to-speech demos to music generators, you'll never run out of stuff to geek out over.

Check out our **[Careers Page]**, because that dream job designing AI to generate haikus while turning photos into watercolor videos? Yeah, it's probably here.  

---

**Team & Enterprise: AI Superpowers for Your Business**  
For organizations that *really* want to flex their AI muscle, we’ve got you covered with our Team and Enterprise subscriptions. It’s everything you love about Hugging Face, supercharged:  
- **Enterprise-grade security**: Sleep easy; your AI models are in safe hands.  
- **Region control & private storage**: Keep your data right where you want it (#DataPrivacyWin).  
- **Analytics & Inference Providers**: Stay on top of your AI game with usage tracking and advanced billing control.  
- **Advanced compute options**: Hello, ZeroGPU quota boost! (Don’t worry; we’ll explain when you get here.)  

Because when it comes to scaling your org's AI efforts, we’re like the wind beneath your models’ wings.  

---

**Let’s Build the Future Together**  
Hugging Face is more than a platform—it’s a movement. Our mission is to empower machine learning professionals, researchers, and curious humans everywhere to collaborate, create, and impact the world in positive ways.  

Because the future of AI doesn’t belong to just one company, one country, or one human. It belongs to all of us—and you're invited.  

So, what do you say? Ready for a hug?  

♥ **Hugging Face**: Bringing intelligence, humanity, and a touch of whimsy to the world of AI. Let's build something amazing together.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>